# Verification of mod 256 classification of eigenforms
For the faster option of using the precomputed source data, set USE_ARCHIVED_SOURCE_DATA = True, Otherwise, set it to FALSE to recompute the source data.

In [1]:
import sys
from pathlib import Path

PYTHON_DIRECTORY = Path("python").resolve()
if str(PYTHON_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(PYTHON_DIRECTORY))

from hecke_congruences import *
from load_source_data import load_source_data

USE_ARCHIVED_SOURCE_DATA = True
SOURCE_DATA_DIRECTORY = Path("source_data")
MOD256_SOURCE_ARCHIVE = (
    SOURCE_DATA_DIRECTORY / "p2_mod256_T3_T5_all_degrees.npz"
)

def get_source_data(R, d, q, archive):
    """Load the archived source when requested; otherwise construct it over R.

    Keep the stated source scope, cyclic orders and orientation.
    """
    if USE_ARCHIVED_SOURCE_DATA:
        return load_source_data(R, d, q, archive)

    return prepare_source_data(R, d, q)

# Identities for $T_3$ and $T_5$

In [2]:
p = 2
m = 8

R = Integers(p^m)
S.<X> = PolynomialRing(R)

period = euler_phi(p^m)

a_m = p^m * (p - 1)          # 256
b_m = p^(m - 1) * (p + 1)    # 384
surjectivity_bound = a_m + b_m  # 640

exact_induction_base = tuple(
    range(a_m, surjectivity_bound, 2)
)

lower_base_degrees = tuple(
    range(0, a_m, 2)
)

degree_residues = tuple(sorted({
    d % period
    for d in exact_induction_base
}))

F5 = {
    r: X - (1 + 5^(r + 1))
    for r in degree_residues
}

epsilon = {
    r: 1
    if r % 16 in (0, 6, 8, 14)
    else 0
    for r in degree_residues
}

Q3 = {
    r: X - (1 + 3^(r + 1))
    for r in degree_residues
}

F3 = {
    r: X^2 - epsilon[r]*X
    for r in degree_residues
}

print("Dickson degrees:", a_m, b_m)
print("degree residues:", degree_residues)
print("epsilon values:", epsilon)
print("lower verification range:", lower_base_degrees)
print("exact induction base:", exact_induction_base)

Dickson degrees: 256 384
degree residues: (0, 2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24, 26, 28, 30, 32, 34, 36, 38, 40, 42, 44, 46, 48, 50, 52, 54, 56, 58, 60, 62, 64, 66, 68, 70, 72, 74, 76, 78, 80, 82, 84, 86, 88, 90, 92, 94, 96, 98, 100, 102, 104, 106, 108, 110, 112, 114, 116, 118, 120, 122, 124, 126)
epsilon values: {0: 1, 2: 0, 4: 0, 6: 1, 8: 1, 10: 0, 12: 0, 14: 1, 16: 1, 18: 0, 20: 0, 22: 1, 24: 1, 26: 0, 28: 0, 30: 1, 32: 1, 34: 0, 36: 0, 38: 1, 40: 1, 42: 0, 44: 0, 46: 1, 48: 1, 50: 0, 52: 0, 54: 1, 56: 1, 58: 0, 60: 0, 62: 1, 64: 1, 66: 0, 68: 0, 70: 1, 72: 1, 74: 0, 76: 0, 78: 1, 80: 1, 82: 0, 84: 0, 86: 1, 88: 1, 90: 0, 92: 0, 94: 1, 96: 1, 98: 0, 100: 0, 102: 1, 104: 1, 106: 0, 108: 0, 110: 1, 112: 1, 114: 0, 116: 0, 118: 1, 120: 1, 122: 0, 124: 0, 126: 1}
lower verification range: (0, 2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24, 26, 28, 30, 32, 34, 36, 38, 40, 42, 44, 46, 48, 50, 52, 54, 56, 58, 60, 62, 64, 66, 68, 70, 72, 74, 76, 78, 80, 82, 84, 86, 88, 90, 92, 94, 96, 

In [3]:
def verify_T3_T5_case(case):
    """
    Verify the divided T3 and ordinary T5 identities in one degree.

    Source data are loaded from the archive or computed afresh,
    according to USE_ARCHIVED_SOURCE_DATA.
    """
    d, q = case

    data = get_source_data(R, d, q, MOD256_SOURCE_ARCHIVE)

    T5_test = verify_ordinary_identities(
        F=F5[d % period],
        n=5,
        data=data,
        check_descent=False,
    )

    r = d % period

    T3_test = verify_divided_identities(
        F=F3[r],
        Q=Q3[r],
        n=3,
        a=7,
        b=1,
        data=data,
        check_descent=False,
    )

    return {
        "T3": T3_test,
        "T5": T5_test,
    }

ordered_degrees = (
    tuple(sorted(exact_induction_base, reverse=True))
    + tuple(sorted(lower_base_degrees, reverse=True))
)

# For p=2, range(0, p-1) contains only q=0.
cases = [
    (d, q)
    for d in ordered_degrees
    for q in range(0, p - 1)
]
results = []

for case in cases:
    d, q = case
    test = verify_T3_T5_case(case)
    results.append(test)

    T3_test = test["T3"]
    T5_test = test["T5"]

    print(
        f"d={d:3d}, "
        f"r={T3_test['residue']:3d}, "
        f"q={q}, "
        f"sign={T3_test['sign']}, "
        f"rank={T3_test['rank']:3d}, "
        f"T3={T3_test['passed']}, "
        f"T3_route={T3_test['verification_route']}, "
        f"T5={T5_test['passed']}"
    )

    assert T3_test["passed"] and T5_test["passed"]

print("cases completed:", len(results))
print("————————————————————————————————————————")
print("IDENTITIES VERIFIED")

d=638, r=126, q=0, sign=None, rank=213, T3=True, T3_route=global_scaled_annihilation, T5=True
d=636, r=124, q=0, sign=None, rank=212, T3=True, T3_route=global_scaled_annihilation, T5=True
d=634, r=122, q=0, sign=None, rank=212, T3=True, T3_route=global_scaled_annihilation, T5=True
d=632, r=120, q=0, sign=None, rank=211, T3=True, T3_route=global_scaled_annihilation, T5=True
d=630, r=118, q=0, sign=None, rank=210, T3=True, T3_route=global_scaled_annihilation, T5=True
d=628, r=116, q=0, sign=None, rank=210, T3=True, T3_route=global_scaled_annihilation, T5=True
d=626, r=114, q=0, sign=None, rank=209, T3=True, T3_route=global_scaled_annihilation, T5=True
d=624, r=112, q=0, sign=None, rank=208, T3=True, T3_route=global_scaled_annihilation, T5=True
d=622, r=110, q=0, sign=None, rank=208, T3=True, T3_route=global_scaled_annihilation, T5=True
d=620, r=108, q=0, sign=None, rank=207, T3=True, T3_route=global_scaled_annihilation, T5=True
d=618, r=106, q=0, sign=None, rank=206, T3=True, T3_route=gl

# Check that each possible signature given by the identities above is realized by a strong eigenform

In [4]:
weight_period = 64
possible_signatures = {}

for k_residue in range(0, weight_period, 2):
    degree_residue = (k_residue - 2) % period
    c3 = R(1 + 3^(degree_residue + 1))
    c5 = R(1 + 5^(degree_residue + 1))

    possible_signatures[k_residue] = {
        (ZZ(c3), ZZ(c5))
    }

    if epsilon[degree_residue] == 1:
        possible_signatures[k_residue].add(
            (ZZ(c3 + 2^7), ZZ(c5))
        )

print(
    f"{'k mod ' + str(weight_period):<10} | "
    f"(a_3, a_5) mod {p^m}"
)
print("-" * 70)

for r in sorted(possible_signatures):
    pairs = ", ".join(
        f"({a3}, {a5})"
        for a3, a5 in sorted(possible_signatures[r])
    )

    print(f"{str(r):<10} | {pairs}")

k mod 64   | (a_3, a_5) mod 256
----------------------------------------------------------------------
0          | (44, 206), (172, 206)
2          | (4, 6), (132, 6)
4          | (28, 126)
6          | (244, 54)
8          | (12, 46), (140, 46)
10         | (100, 102), (228, 102)
12         | (252, 222)
14         | (212, 150)
16         | (108, 142), (236, 142)
18         | (68, 198), (196, 198)
20         | (220, 62)
22         | (180, 246)
24         | (76, 238), (204, 238)
26         | (36, 38), (164, 38)
28         | (188, 158)
30         | (148, 86)
32         | (44, 78), (172, 78)
34         | (4, 134), (132, 134)
36         | (156, 254)
38         | (116, 182)
40         | (12, 174), (140, 174)
42         | (100, 230), (228, 230)
44         | (124, 94)
46         | (84, 22)
48         | (108, 14), (236, 14)
50         | (68, 70), (196, 70)
52         | (92, 190)
54         | (52, 118)
56         | (76, 110), (204, 110)
58         | (36, 166), (164, 166)
60         | (60, 30)


In [5]:
def krw_reduction_to_integer(x, nf, pr, p, m):
    """Find an integer representative of the selected coefficient modulo the KRW ideal.

    Test equality in the prime-ideal quotient rather than assuming the residue is rational.
    """
    e = pr[2]
    N = e*(m - 1) + 1

    candidates = [
        r
        for r in range(p^m)
        if nf.idealval(x - r, pr) >= N
    ]

    if len(candidates) != 1:
        raise ValueError(
            f"expected one residue in Z/{p^m}Z, "
            f"but found {candidates}"
        )

    return Integers(p^m)(candidates[0])

In [6]:
B = 90
p = 2
m = 8
l1, l2 = 3, 5

R = Integers(p^m)
signatures = []

for k in range(12, B + 1, 2):
    S = CuspForms(1, k)

    if S.dimension() == 0:
        continue

    print(f"calculating strong signatures at weight {k}")

    for j, f in enumerate(S.newforms(names='a')):
        K = f.base_ring()

        # Rational eigenform orbit
        if K == QQ:
            signature = (
                k,
                R(f[l1]),
                R(f[l2]),
            )
            signatures.append(signature)
            print(signature)
            continue

        # Nonrational eigenform orbit
        pol = K.pari_polynomial('y')
        nf = pari([pol, [p]]).nfinit(4)

        a1 = pari(f[l1])
        a2 = pari(f[l2])

        for pr in nf.idealprimedec(p):
            a1_bar = krw_reduction_to_integer(
                a1, nf, pr, p, m
            )

            a2_bar = krw_reduction_to_integer(
                a2, nf, pr, p, m
            )

            signature = (k, a1_bar, a2_bar)
            signatures.append(signature)

calculating strong signatures at weight 12
(12, 252, 222)
calculating strong signatures at weight 16
(16, 236, 142)
calculating strong signatures at weight 18
(18, 68, 198)
calculating strong signatures at weight 20
(20, 220, 62)
calculating strong signatures at weight 22
(22, 180, 246)
calculating strong signatures at weight 24
calculating strong signatures at weight 26
(26, 36, 38)
calculating strong signatures at weight 28
calculating strong signatures at weight 30
calculating strong signatures at weight 32
calculating strong signatures at weight 34
calculating strong signatures at weight 36
calculating strong signatures at weight 38
calculating strong signatures at weight 40
calculating strong signatures at weight 42
calculating strong signatures at weight 44
calculating strong signatures at weight 46
calculating strong signatures at weight 48
calculating strong signatures at weight 50
calculating strong signatures at weight 52
calculating strong signatures at weight 54
calculating

In [7]:
signatures_by_weight_residue = {}

for k, a3, a5 in signatures:
    r = k % weight_period

    if r not in signatures_by_weight_residue:
        signatures_by_weight_residue[r] = set()

    signatures_by_weight_residue[r].add((ZZ(a3), ZZ(a5)))

In [8]:
print(
    f"{'k mod ' + str(weight_period):<10} | "
    f"(a_{l1}, a_{l2}) mod {p^m}"
)
print("-" * 70)

for r in sorted(signatures_by_weight_residue):
    pairs = ", ".join(
        f"({a1}, {a2})"
        for a1, a2 in sorted(signatures_by_weight_residue[r])
    )

    print(f"{str(r):<10} | {pairs}")

k mod 64   | (a_3, a_5) mod 256
----------------------------------------------------------------------
0          | (44, 206), (172, 206)
2          | (4, 6), (132, 6)
4          | (28, 126)
6          | (244, 54)
8          | (12, 46), (140, 46)
10         | (100, 102), (228, 102)
12         | (252, 222)
14         | (212, 150)
16         | (108, 142), (236, 142)
18         | (68, 198), (196, 198)
20         | (220, 62)
22         | (180, 246)
24         | (76, 238), (204, 238)
26         | (36, 38), (164, 38)
28         | (188, 158)
30         | (148, 86)
32         | (44, 78), (172, 78)
34         | (4, 134), (132, 134)
36         | (156, 254)
38         | (116, 182)
40         | (12, 174), (140, 174)
42         | (100, 230), (228, 230)
44         | (124, 94)
46         | (84, 22)
48         | (108, 14), (236, 14)
50         | (68, 70), (196, 70)
52         | (92, 190)
54         | (52, 118)
56         | (76, 110), (204, 110)
58         | (36, 166), (164, 166)
60         | (60, 30)


In [9]:
assert possible_signatures == signatures_by_weight_residue

print("EVERY POSSIBLE MODULO-256 SIGNATURE IS REALIZED")

EVERY POSSIBLE MODULO-256 SIGNATURE IS REALIZED
